# ForestWatch Papua - Full Pipeline

Notebook tunggal yang menjalankan **seluruh tahap** dari [PRD v2.0 §A](../../docs/PRD_ForestWatch_Papua_v2.md) dengan modul `forestwatch.*`.

**Alur (urut Minggu 1 -> Minggu 4):**
1. Setup (install, clone, mount Drive)
2. (Opsional) Generate dummy 7 file untuk Orang 2 - **kirim di akhir Minggu 1**
3. Auth GEE + komposit Sentinel-2 + label fusion 6 aturan
4. Ekspor 36 ubin T1 (2021) + 36 ubin T2 (2025) ke Drive
5. Cut patches 256x256 dari ubin
6. Build ResNet50-U-Net + train dengan AMP + early stopping
7. Plot training history
8. Evaluasi pada test set + confusion matrix
9. Ekspor model.onnx
10. Inferensi T1 + T2 (72 ubin -> 72 mask)
11. Deteksi 4 transisi + generate 7 file kontrak + validasi schema
12. Visualisasi hasil + ringkasan angka untuk Orang 3 (esai)

**Target metrik (PRD §A.1):** mIoU >= 0,60 (ideal 0,75), IoU Sawit & Deforestasi >= 0,55.

**Catatan:** beberapa cell akan jalan lama (ekspor GEE ~2-4 jam, training ~1-2 jam). Cell yang berat sudah di-flag pada heading.

---

In [ ]:
!git -C /content/fw_repo pull


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.06 KiB | 217.00 KiB/s, done.
From https://github.com/Ridho-Dwi-Syahputra/forestwatch-model
   c41e333..f9a3e92  main       -> origin/main
Updating c41e333..f9a3e92
Fast-forward
 notebooks/forestwatch_papua_full_pipeline.ipynb | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


## Bagian 0 - Setup

Jalankan **satu kali** di awal sesi Colab. Untuk lokal Windows, comment baris clone/install/mount Drive.

In [2]:
# === COLAB SETUP ===
!git clone https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo
%cd fw_repo
!pip install -q -e ".[gee,gis,ml]"

# Pastikan src package yang terpakai, bukan folder repo
import sys, importlib
if '/content/fw_repo/src' not in sys.path:
    sys.path.insert(0, '/content/fw_repo/src')
# Bersihkan cache import yang salah
for mod in list(sys.modules.keys()):
    if mod == 'forestwatch' or mod.startswith('forestwatch.'):
        del sys.modules[mod]
importlib.invalidate_caches()

print('Python:', sys.version)

try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('torch belum ter-install.')

import forestwatch
print(f'forestwatch v{forestwatch.__version__} - {len(forestwatch.CLASS_NAMES)} kelas, {len(forestwatch.BANDS)} band Sentinel-2')


Cloning into 'fw_repo'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 191 (delta 63), reused 171 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (191/191), 114.88 KiB | 2.13 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/fw_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 88.9 MB/s eta 0:00:00
  Building editable for forestwat

In [3]:
# === MOUNT GOOGLE DRIVE (Colab) ===
from google.colab import drive
drive.mount('/content/drive')
print('Drive ter-mount di /content/drive/MyDrive')

Mounted at /content/drive
Drive ter-mount di /content/drive/MyDrive


In [4]:
# === KONFIGURASI PATH ===
# Semua folder ForestWatch_* dibuat di dalam folder 'Satria Data 2.0' di Drive.
from pathlib import Path

# Colab (default):
DRIVE_ROOT = Path('/content/drive/MyDrive/Satria Data 2.0')
# Lokal Windows alternatif:
# DRIVE_ROOT = Path(r'D:/Local Disk D/Tugas/SATRIA DATA/model/data')

TILES_T1   = DRIVE_ROOT / 'ForestWatch_Tiles_T1'   # 36 GeoTIFF citra 2021
TILES_T2   = DRIVE_ROOT / 'ForestWatch_Tiles_T2'   # 36 GeoTIFF citra 2025 + label
PATCH_DIR  = DRIVE_ROOT / 'ForestWatch_Patches'    # patch .npz training + best_model.pt
MASK_DIR   = DRIVE_ROOT / 'ForestWatch_Masks'      # 72 mask hasil inferensi
OUT_DIR    = DRIVE_ROOT / 'ForestWatch_Outputs'    # 7 file kontrak untuk Orang 2
CKPT_PATH  = PATCH_DIR / 'best_model.pt'

for d in (TILES_T1, TILES_T2, PATCH_DIR, MASK_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Folder siap:')
for d in (TILES_T1, TILES_T2, PATCH_DIR, MASK_DIR, OUT_DIR):
    print(' -', d)

Folder siap:
 - /content/drive/MyDrive/Satria Data 2.0/ForestWatch_Tiles_T1
 - /content/drive/MyDrive/Satria Data 2.0/ForestWatch_Tiles_T2
 - /content/drive/MyDrive/Satria Data 2.0/ForestWatch_Patches
 - /content/drive/MyDrive/Satria Data 2.0/ForestWatch_Masks
 - /content/drive/MyDrive/Satria Data 2.0/ForestWatch_Outputs


In [5]:
# Load konfigurasi default (hyperparameters dari configs/default.yaml)
from forestwatch.config import load_config

cfg = load_config()
print(f"Project    : {cfg['project']['name']}")
print(f"Periode    : T1={cfg['periods']['t1']}, T2={cfg['periods']['t2']}")
print(f"Arsitektur : {cfg['model']['architecture']} + {cfg['model']['encoder_name']}")
print(f"Training   : batch={cfg['training']['batch_size']}, epochs={cfg['training']['epochs']}, lr={cfg['training']['learning_rate']}")
print(f"Patch size : {cfg['patches']['size']}")
print(f"Tiles grid : {cfg['export']['tiles_nx']} x {cfg['export']['tiles_ny']} = {cfg['export']['tiles_nx']*cfg['export']['tiles_ny']} ubin")

Project    : ForestWatch Papua
Periode    : T1=2021, T2=2025
Arsitektur : unet + resnet50
Training   : batch=8, epochs=50, lr=0.001
Patch size : 256
Tiles grid : 6 x 6 = 36 ubin


---
## Bagian 1 - (Opsional, **PENTING DI MINGGU 1**) Generate Dummy untuk Orang 2

Sambil menunggu ekspor GEE (~2-4 jam), kirim 7 file dummy schema-valid ke Orang 2 supaya UI WebGIS bisa dibangun paralel. Lihat [Master Plan Model §5](../../docs/model/MASTER_PLAN.md).

**Deadline:** akhir Minggu 1.

In [ ]:
from forestwatch.cli.dummy import main as dummy_main

dummy_out = (DRIVE_ROOT / 'ForestWatch_Outputs_Dummy').as_posix()
dummy_main(['--out', dummy_out, '--n-polygons', '60', '--seed', '42'])

# Validasi dummy sesuai schema PRD §B.1
from forestwatch.validation.schema import validate_outputs_dir
report = validate_outputs_dir(dummy_out)
print(report.render())

2026-05-30 08:21:52 [INFO] forestwatch.outputs.png: PNG landcover disimpan: /content/drive/MyDrive/Satria Data 2.0/ForestWatch_Outputs_Dummy/landcover_2025.png (shape=(512, 512, 4))
2026-05-30 08:21:52 [INFO] forestwatch.outputs.png: PNG landcover disimpan: /content/drive/MyDrive/Satria Data 2.0/ForestWatch_Outputs_Dummy/landcover_2021.png (shape=(512, 512, 4))
{
  "ok": true,
  "out_dir": "/content/drive/MyDrive/Satria Data 2.0/ForestWatch_Outputs_Dummy",
  "files": [
    "deforestation.geojson",
    "landcover_2021.png",
    "landcover_2021_bounds.json",
    "landcover_2025.png",
    "landcover_2025_bounds.json",
    "legend.json",
    "metrics.json",
    "model_card.md",
    "statistics.json"
  ],
  "n_features": 60
}
Files checked: 9
  - landcover_2021.png
  - landcover_2021_bounds.json
  - landcover_2025.png
  - landcover_2025_bounds.json
  - deforestation.geojson
  - statistics.json
  - legend.json
  - metrics.json
  - model_card.md

Warnings (1):
  ! File opsional hilang: model.

**Aksi manual:** zip folder `ForestWatch_Outputs_Dummy/` lalu kirim ke Orang 2, ATAU share folder Drive.

---

## Bagian 2 - Auth GEE + Komposit Sentinel-2 + Label Fusion (Minggu 1)

Pertama kali akan trigger browser auth GEE (sekali per session). Project di-set ke `forestwatch-papua-2` (EECU fresh).

In [ ]:
from forestwatch.gee.auth import init_ee

# Pakai project dengan EECU tersedia. Ganti kalau project lain.
init_ee(project="forestwatch-papua-2")

2026-05-30 08:22:38 [INFO] forestwatch.gee.auth: GEE siap (project=forestwatch-papua-2).


In [ ]:
# Bangun komposit S2 T1 (2021) + T2 (2025) dan label 6 kelas dengan label fusion 6 aturan (PRD §A.4)
import ee
from forestwatch.constants import PAPUA_BBOX
from forestwatch.gee.composite import s2_composite
from forestwatch.gee.label_fusion import build_label

papua = ee.Geometry.Rectangle(list(PAPUA_BBOX))

img_t2 = s2_composite(cfg['periods']['t2'], papua)
img_t1 = s2_composite(cfg['periods']['t1'], papua)
label  = build_label(papua, cfg['periods']['t2'])

# .toFloat() WAJIB: img_t2 (Float32) + label (Byte) bikin error GEE 'inconsistent types'.
stack_t2 = img_t2.addBands(label.toFloat())
print('Komposit S2 (T1 & T2) + label 6 kelas siap.')

Komposit S2 (T1 & T2) + label 6 kelas siap.


In [ ]:
# (Opsional) Verifikasi visual 1 ubin uji di sekitar Merauke pakai geemap
# Uncomment jika geemap terinstall (sudah include di [gee])

import geemap
Map = geemap.Map(center=[-8.5, 140.4], zoom=9)
Map.addLayer(stack_t2.select(['B4','B3','B2']), {'min':0,'max':0.3}, 'S2 RGB 2025')
Map.addLayer(label, {'min':0,'max':5,
                      'palette':['2A6FDB','0B3D0B','E03B24','F97316','E9C46A','6D4C41']}, 'Label')
Map

Map(center=[-8.5, 140.4], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

---
## Bagian 3 - Ekspor 36 ubin T1 + T2 ke Drive (Minggu 1-2)

**Cell ini start 72 task batch ekspor di server GEE.** Total durasi ~2-4 jam (laptop boleh ditutup). Pantau status di [https://code.earthengine.google.com/tasks](https://code.earthengine.google.com/tasks).

In [ ]:
from forestwatch.gee.tiles import make_tiles
from forestwatch.gee.export import export_tiles_grid

tiles = make_tiles(papua, nx=cfg['export']['tiles_nx'], ny=cfg['export']['tiles_ny'])
print(f'Total tiles: {len(tiles)}')

# T2: citra + label (untuk training)
tasks_t2 = export_tiles_grid(
    stack_t2, tiles,
    name_prefix='papua_t2_tile',
    folder=TILES_T2.name,
    scale=cfg['sentinel2']['scale'],
    max_pixels=int(cfg['export']['max_pixels']),
)

# T1: hanya citra (untuk inferensi T1, lalu deteksi perubahan)
tasks_t1 = export_tiles_grid(
    img_t1, tiles,
    name_prefix='papua_t1_tile',
    folder=TILES_T1.name,
    scale=cfg['sentinel2']['scale'],
    max_pixels=int(cfg['export']['max_pixels']),
)
print(f'Started {len(tasks_t1) + len(tasks_t2)} task. Pantau di GEE Tasks.')

Total tiles: 36
2026-05-30 08:23:25 [INFO] forestwatch.gee.export: Task 'papua_t2_tile_00' dimulai (folder=ForestWatch_Tiles_T2).
2026-05-30 08:23:26 [INFO] forestwatch.gee.export: Task 'papua_t2_tile_01' dimulai (folder=ForestWatch_Tiles_T2).
2026-05-30 08:23:26 [INFO] forestwatch.gee.export: Task 'papua_t2_tile_02' dimulai (folder=ForestWatch_Tiles_T2).
2026-05-30 08:23:26 [INFO] forestwatch.gee.export: Task 'papua_t2_tile_03' dimulai (folder=ForestWatch_Tiles_T2).
2026-05-30 08:23:27 [INFO] forestwatch.gee.export: Task 'papua_t2_tile_04' dimulai (folder=ForestWatch_Tiles_T2).
2026-05-30 08:23:27 [INFO] forestwatch.gee.export: Task 'papua_t2_tile_05' dimulai (folder=ForestWatch_Tiles_T2).
2026-05-30 08:23:28 [INFO] forestwatch.gee.export: Task 'papua_t2_tile_06' dimulai (folder=ForestWatch_Tiles_T2).
2026-05-30 08:23:28 [INFO] forestwatch.gee.export: Task 'papua_t2_tile_07' dimulai (folder=ForestWatch_Tiles_T2).
2026-05-30 08:23:28 [INFO] forestwatch.gee.export: Task 'papua_t2_tile_0

In [ ]:
# (Opsional) Monitor sampai semua task selesai - polling tiap 60 detik (max 6 jam).
# Jangan jalankan jika ingin pakai Colab untuk hal lain - cukup buka GEE Tasks console.

# from forestwatch.gee.export import monitor_tasks
# final_status = monitor_tasks(tasks_t1 + tasks_t2, poll_seconds=60, timeout_seconds=6*3600)
# print('Status akhir:', final_status)

**Tunggu sampai semua task ter-export.** Setelah selesai, semua GeoTIFF muncul di Drive:
- `ForestWatch_Tiles_T2/papua_t2_tile_XX.tif` (7 band = 6 citra + 1 label)
- `ForestWatch_Tiles_T1/papua_t1_tile_XX.tif` (6 band citra)

Lanjutkan ke Bagian 4 setelah selesai.

---

## Bagian 4 - Cut Patches 256x256 (Minggu 2)

Potong 36 ubin T2 menjadi patch 256x256 `.npz`. Buang patch dengan >30% NaN (akibat awan).

In [6]:
# ── Step 1: Update list_patches agar bisa baca subfolder per-tile ────────────
import sys, importlib
from pathlib import Path

patches_file = Path('/content/fw_repo/src/forestwatch/data/patches.py')
code = patches_file.read_text()

old = '    return sorted(Path(patch_dir).glob("p*.npz"))'
new = (
    '    flat = sorted(Path(patch_dir).glob("p*.npz"))\n'
    '    if flat:\n'
    '        return flat\n'
    '    return sorted(Path(patch_dir).rglob("p*.npz"))'
)
if old in code:
    patches_file.write_text(code.replace(old, new))
    print("list_patches diupdate — support flat + per-tile subfolder")
else:
    print("Sudah diupdate sebelumnya")

for mod in list(sys.modules.keys()):
    if 'forestwatch' in mod:
        del sys.modules[mod]
importlib.invalidate_caches()
print("Module reloaded.")

list_patches diupdate — support flat + per-tile subfolder
Module reloaded.


In [7]:
# ── Step 2: Cut patches RESILIENT (lokal-first + verifikasi EXACT + _DONE) ───
# Mengatasi error Drive putus ('Transport endpoint is not connected') DAN
# tile yang salah ditandai selesai. Kunci: _DONE hanya ditulis kalau jumlah
# file di Drive PERSIS SAMA dengan jumlah patch lokal yang dipotong.
# CATATAN: _DONE=0 valid untuk tile LAUT (semua NaN) → di-skip, tidak di-cut ulang.
import numpy as np
import rasterio
import shutil
import time
from rasterio.windows import Window
from pathlib import Path
from tqdm.auto import tqdm


def _remount_drive():
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(3)
    drive.mount('/content/drive', force_remount=True)
    time.sleep(2)
    print("  -> Drive di-remount.")


def _read_done_count(done_marker):
    """Baca jumlah patch yang tercatat di penanda _DONE (int), atau None."""
    try:
        if done_marker.exists():
            return int(done_marker.read_text().strip())
    except (OSError, ValueError):
        return None
    return None


def cut_patches_resilient(tile_dir, patch_base_dir, *,
                          patch_size=256, stride=256, max_nan_ratio=0.3,
                          n_channels_image=6, local_tmp='/content/_tmp_patches'):
    tile_files = sorted(Path(tile_dir).glob('*.tif'))
    if not tile_files:
        raise FileNotFoundError(f"Tidak ada .tif di {tile_dir}")

    patch_base = Path(patch_base_dir)
    local_root = Path(local_tmp)
    n_tiles = len(tile_files)
    total = 0

    for ti, tif in enumerate(tile_files):
        drive_out = patch_base / f'tile_{ti:03d}'
        done_marker = drive_out / '_DONE'
        header = f"Tile {ti+1}/{n_tiles}  (tile_{ti:03d})"

        # --- SKIP kalau _DONE valid DAN jumlah file cocok (termasuk 0 utk laut) ---
        try:
            done_n = _read_done_count(done_marker)
            actual_n = len(list(drive_out.glob('p*.npz'))) if drive_out.exists() else 0
        except OSError:
            _remount_drive()
            done_n = _read_done_count(done_marker)
            actual_n = len(list(drive_out.glob('p*.npz'))) if drive_out.exists() else 0

        if done_n is not None and actual_n == done_n:   # 0==0 (laut) juga di-skip
            total += actual_n
            tag = "laut/kosong" if done_n == 0 else f"{actual_n} patch"
            print(f"{header}: SKIP - komplit ({tag})")
            continue

        # --- Folder partial / tidak konsisten -> hapus, cut ulang ---
        try:
            if drive_out.exists():
                shutil.rmtree(drive_out, ignore_errors=True)
        except OSError:
            _remount_drive()
            shutil.rmtree(drive_out, ignore_errors=True)

        # --- Potong patch ke disk LOKAL ---
        ltile = local_root / f'tile_{ti:03d}'
        if ltile.exists():
            shutil.rmtree(ltile)
        ltile.mkdir(parents=True, exist_ok=True)

        idx = 0
        with rasterio.open(tif) as src:
            W, H, nb = src.width, src.height, src.count
            row_list = list(range(0, H - patch_size + 1, stride))
            col_list = list(range(0, W - patch_size + 1, stride))
            pbar = tqdm(total=len(row_list) * len(col_list), desc=header, unit="win", leave=True)
            for r in row_list:
                for c in col_list:
                    arr = src.read(window=Window(c, r, patch_size, patch_size))
                    pbar.update(1)
                    if arr.shape != (nb, patch_size, patch_size):
                        continue
                    img = arr[:n_channels_image].astype('float32')
                    if np.isnan(img).mean() > max_nan_ratio:
                        continue
                    img = np.nan_to_num(img)
                    kw = dict(img=img, tile=tif.name, row=r, col=c)
                    if nb > n_channels_image:
                        kw['lab'] = np.nan_to_num(arr[n_channels_image], nan=0.0).astype('uint8')
                    np.savez_compressed(ltile / f'p{idx:05d}.npz', **kw)
                    idx += 1
                    pbar.set_postfix(patch=idx)
            pbar.close()

        local_files = sorted(ltile.glob('p*.npz'))
        assert len(local_files) == idx, "jumlah file lokal tidak konsisten"

        # --- Sync LOKAL -> Drive dengan verifikasi EXACT (==) ---
        synced = False
        for attempt in range(1, 8):
            try:
                drive_out.mkdir(parents=True, exist_ok=True)
                for f in local_files:
                    dst = drive_out / f.name
                    if (not dst.exists()) or (dst.stat().st_size != f.stat().st_size):
                        shutil.copy2(f, dst)
                drive_n = len(list(drive_out.glob('p*.npz')))
                if drive_n == idx:                       # HARUS sama persis (0 utk laut OK)
                    done_marker.write_text(str(idx))
                    synced = True
                    break
                print(f"  verifikasi belum cocok: Drive={drive_n} vs lokal={idx} (attempt {attempt})")
            except OSError as e:
                print(f"  sync gagal (attempt {attempt}): {e}")
                _remount_drive()
            time.sleep(2)
        shutil.rmtree(ltile, ignore_errors=True)

        if not synced:
            print(f"{header}: GAGAL verifikasi sync - STOP. Hapus folder tile ini di Drive lalu jalankan ulang.")
            return total

        total += idx
        tag = "LAUT/kosong (0 patch)" if idx == 0 else f"{idx} patch"
        print(f"{header}: SELESAI - {tag} -> Drive/tile_{ti:03d}/\n")

    print(f"=== SEMUA TILE SELESAI. Total {total} patch ===")
    return total

print("Fungsi cut_patches_resilient siap (skip _DONE termasuk laut 0 patch).")

Fungsi cut_patches_resilient siap (skip _DONE termasuk laut 0 patch).


In [ ]:
# from collections import Counter, defaultdict
# import shutil

# # 1. Kumpulkan jumlah file per tile, kelompokkan per posisi (idx % 4)
# counts, by_pos = {}, defaultdict(list)
# for ti in range(144):
#     d = PATCH_DIR / f"tile_{ti:03d}"
#     if d.exists():
#         n = len(list(d.glob("p*.npz")))
#         counts[ti] = n
#         by_pos[ti % 4].append(n)

# # 2. Jumlah "normal" per posisi = nilai paling sering muncul (mode)
# expected = {pos: Counter(v).most_common(1)[0][0] for pos, v in by_pos.items()}
# print("Jumlah normal per posisi (idx%4):", expected)

# # 3. Bless yang cocok, hapus + flag yang outlier (untuk di-cut ulang)
# blessed, flagged = 0, []
# for ti, n in counts.items():
#     d = PATCH_DIR / f"tile_{ti:03d}"
#     exp = expected[ti % 4]
#     if n == exp and n > 0:
#         (d / "_DONE").write_text(str(n))
#         blessed += 1
#     else:
#         flagged.append((ti, n, exp))
#         shutil.rmtree(d, ignore_errors=True)

# print(f"\nDi-bless komplit : {blessed}")
# print(f"Outlier dihapus  : {len(flagged)} (akan di-cut ulang oleh Step 3)")
# for ti, n, exp in flagged:
#     print(f"  tile_{ti:03d}: {n} (normal {exp})")


Jumlah normal per posisi (idx%4): {0: 2401, 1: 1568, 2: 1127, 3: 736}

Di-bless komplit : 92
Outlier dihapus  : 27 (akan di-cut ulang oleh Step 3)
  tile_002: 1078 (normal 1127)
  tile_003: 726 (normal 736)
  tile_037: 1564 (normal 1568)
  tile_068: 2216 (normal 2401)
  tile_069: 895 (normal 1568)
  tile_084: 2400 (normal 2401)
  tile_085: 1563 (normal 1568)
  tile_087: 731 (normal 736)
  tile_088: 2398 (normal 2401)
  tile_092: 909 (normal 2401)
  tile_093: 102 (normal 1568)
  tile_095: 642 (normal 736)
  tile_097: 1566 (normal 1568)
  tile_100: 2397 (normal 2401)
  tile_101: 1541 (normal 1568)
  tile_103: 735 (normal 736)
  tile_104: 2400 (normal 2401)
  tile_105: 1460 (normal 1568)
  tile_107: 726 (normal 736)
  tile_108: 2386 (normal 2401)
  tile_109: 1552 (normal 1568)
  tile_110: 1124 (normal 1127)
  tile_111: 733 (normal 736)
  tile_114: 1123 (normal 1127)
  tile_116: 0 (normal 2401)
  tile_117: 0 (normal 1568)
  tile_118: 573 (normal 1127)


In [11]:
N_TILES = 144
ok, ocean, partial, missing = [], [], [], []
for ti in range(N_TILES):
    d = PATCH_DIR / f"tile_{ti:03d}"
    if not d.exists():
        missing.append(ti); continue
    n = len(list(d.glob('p*.npz')))
    dn = None
    if (d/'_DONE').exists():
        try: dn = int((d/'_DONE').read_text().strip())
        except: dn = None
    if dn is not None and dn == n:
        (ocean if n == 0 else ok).append(ti)
    else:
        partial.append((f"tile_{ti:03d}", n, dn))
print(f"Berisi data : {len(ok)}")
print(f"Laut (0)    : {len(ocean)} → {ocean}")
print(f"Partial     : {len(partial)} → {partial}")
print(f"Belum ada   : {len(missing)}")
print(f"SELESAI     : {len(ok)+len(ocean)} / {N_TILES}")


Berisi data : 139
Laut (0)    : 5 → [116, 117, 140, 141, 143]
Partial     : 0 → []
Belum ada   : 0
SELESAI     : 144 / 144


In [10]:
# ── Step 3: Hapus patch flat lama lalu jalankan (resilient) ──────────────────
# Hapus patch flat lama (p00000.npz langsung di PATCH_DIR) kalau ada
for f in PATCH_DIR.glob("p*.npz"):
    f.unlink()
print(f"Patch flat tersisa: {len(list(PATCH_DIR.glob('p*.npz')))} (harus 0)")

# Jalankan. Kalau Drive putus / session mati di tengah jalan:
#   cukup jalankan ULANG cell ini — tile yang sudah ada di Drive di-SKIP otomatis.
n_total = cut_patches_resilient(
    tile_dir=TILES_T2,
    patch_base_dir=PATCH_DIR,
    patch_size=cfg['patches']['size'],
    stride=cfg['patches']['stride'],
    max_nan_ratio=cfg['patches']['max_nan_ratio'],
)
print(f'\nTotal patch: {n_total}')

Patch flat tersisa: 0 (harus 0)
Tile 1/144  (tile_000): SKIP - komplit (2401 patch)
Tile 2/144  (tile_001): SKIP - komplit (1568 patch)
Tile 3/144  (tile_002): SKIP - komplit (1078 patch)
Tile 4/144  (tile_003): SKIP - komplit (726 patch)
Tile 5/144  (tile_004): SKIP - komplit (2401 patch)
Tile 6/144  (tile_005): SKIP - komplit (1568 patch)
Tile 7/144  (tile_006): SKIP - komplit (1127 patch)
Tile 8/144  (tile_007): SKIP - komplit (736 patch)
Tile 9/144  (tile_008): SKIP - komplit (2401 patch)
Tile 10/144  (tile_009): SKIP - komplit (1568 patch)
Tile 11/144  (tile_010): SKIP - komplit (1127 patch)
Tile 12/144  (tile_011): SKIP - komplit (736 patch)
Tile 13/144  (tile_012): SKIP - komplit (2401 patch)
Tile 14/144  (tile_013): SKIP - komplit (1568 patch)
Tile 15/144  (tile_014): SKIP - komplit (1127 patch)
Tile 16/144  (tile_015): SKIP - komplit (736 patch)
Tile 17/144  (tile_016): SKIP - komplit (2401 patch)
Tile 18/144  (tile_017): SKIP - komplit (1568 patch)
Tile 19/144  (tile_018): SK

Tile 137/144  (tile_136):   0%|          | 0/2401 [00:00<?, ?win/s]

Tile 137/144  (tile_136): SELESAI - 2373 patch -> Drive/tile_136/

Tile 138/144  (tile_137): SKIP - komplit (1330 patch)


Tile 139/144  (tile_138):   0%|          | 0/1127 [00:00<?, ?win/s]

Tile 139/144  (tile_138): SELESAI - 1127 patch -> Drive/tile_138/

Tile 140/144  (tile_139): SKIP - komplit (736 patch)
Tile 141/144  (tile_140): SKIP - komplit (laut/kosong)
Tile 142/144  (tile_141): SKIP - komplit (laut/kosong)
Tile 143/144  (tile_142): SKIP - komplit (80 patch)
Tile 144/144  (tile_143): SKIP - komplit (laut/kosong)
=== SEMUA TILE SELESAI. Total 193453 patch ===

Total patch: 193453


---
## Bagian 4b - Deteksi & Perbaiki Tile Bermasalah (Korup vs Laut)

Dua cell di bawah memastikan dataset **benar-benar lengkap**:
1. **Cell deteksi** — bedakan tile yang KORUP (file rusak, harus re-export) vs LAUT (memang 0 patch karena di luar daratan Papua, biarkan).
2. **Cell perbaikan** — re-export HANYA original tile yang korup dari GEE, timpa file lama di Drive, hapus penanda `_DONE` sub-tile terkait agar di-cut ulang.

In [ ]:
# ── Cell 4b-1: DETEKSI korup vs laut (TARGETED — cuma cek tile anomali) ──────
# Kunci kecepatan: jumlah patch dihitung dari folder (INSTAN, tanpa baca GeoTIFF).
# File GeoTIFF cuma dibaca penuh untuk SUSPECT (patch < 50% normal) — segelintir saja.
import numpy as np
import rasterio

tile_files = sorted(TILES_T2.glob("*.tif"))
EXPECTED = {0: 2401, 1: 1568, 2: 1127, 3: 736}

# 1. Cari suspect dari jumlah patch (INSTAN, tidak baca GeoTIFF) ─────────────
suspects = []
for ti in range(len(tile_files)):
    d = PATCH_DIR / f"tile_{ti:03d}"
    n = len(list(d.glob("p*.npz"))) if d.exists() else 0
    if n < EXPECTED[ti % 4] * 0.5:          # < 50% normal = anomali berat
        suspects.append((ti, n, EXPECTED[ti % 4]))

print(f"Suspect (patch < 50% normal): {[(s[0], s[1]) for s in suspects]}\n")

# 2. Verifikasi HANYA suspect: baca penuh band 1 → korup (error) atau laut (NaN) ─
corrupt_subtiles, ocean_subtiles, low_subtiles = [], [], []
for ti, n, exp in suspects:
    f = tile_files[ti]
    try:
        with rasterio.open(f) as src:
            band = src.read(1)               # baca penuh → error kalau blok korup
            nan_ratio = float(np.isnan(band).mean())
        if nan_ratio > 0.95:
            ocean_subtiles.append(ti)
            print(f"tile_{ti:03d}: LAUT 🌊   {n}/{exp} patch, {nan_ratio:.0%} NaN - biarkan")
        else:
            low_subtiles.append(ti)
            print(f"tile_{ti:03d}: RENDAH ⚠️ {n}/{exp} patch, {nan_ratio:.0%} NaN - cek manual")
    except Exception as e:
        corrupt_subtiles.append(ti)
        print(f"tile_{ti:03d}: KORUP ❌  {n}/{exp} patch - {str(e)[:50]}")

print("\n" + "=" * 55)
print(f"KORUP (re-export) : {corrupt_subtiles}")
print(f"LAUT  (biarkan)   : {ocean_subtiles}")
print(f"RENDAH (cek)      : {low_subtiles}")
corrupt_original = sorted(set(ti // 4 for ti in corrupt_subtiles))
print(f"\nOriginal tile GEE perlu re-export: {corrupt_original}")

Suspect (patch < 50% normal): [(92, 909), (93, 102), (116, 0), (117, 0), (118, 0), (119, 118), (140, 0), (141, 0), (142, 80), (143, 0)]

tile_092: RENDAH ⚠️ 909/2401 patch, 62% NaN - cek manual
tile_093: RENDAH ⚠️ 102/1568 patch, 93% NaN - cek manual
tile_116: LAUT 🌊   0/2401 patch, 100% NaN - biarkan
tile_117: LAUT 🌊   0/1568 patch, 100% NaN - biarkan
tile_118: RENDAH ⚠️ 0/1127 patch, 47% NaN - cek manual
tile_119: RENDAH ⚠️ 118/736 patch, 80% NaN - cek manual
tile_140: LAUT 🌊   0/2401 patch, 100% NaN - biarkan
tile_141: LAUT 🌊   0/1568 patch, 100% NaN - biarkan
tile_142: RENDAH ⚠️ 80/1127 patch, 91% NaN - cek manual
tile_143: LAUT 🌊   0/736 patch, 100% NaN - biarkan

KORUP (re-export) : []
LAUT  (biarkan)   : [116, 117, 140, 141, 143]
RENDAH (cek)      : [92, 93, 118, 119, 142]

Original tile GEE perlu re-export: []


In [ ]:
# ── Cell 4b-2: RE-EXPORT tile korup + hapus file lama + reset _DONE ──────────
# Jalankan SETELAH cell deteksi (butuh variabel corrupt_original & corrupt_subtiles).
import shutil
import ee
from forestwatch.gee.auth import init_ee
from forestwatch.gee.composite import s2_composite
from forestwatch.gee.label_fusion import build_label
from forestwatch.gee.tiles import make_tiles
from forestwatch.gee.export import export_stack
from forestwatch.constants import PAPUA_BBOX

if not corrupt_original:
    print("Tidak ada tile korup. Tidak perlu re-export. Lanjut ke distribusi kelas.")
else:
    print(f"Akan re-export {len(corrupt_original)} original tile: {corrupt_original}\n")

    # 1. Auth + rebuild stack (sama persis Bagian 2)
    init_ee(project="forestwatch-papua-2")
    papua = ee.Geometry.Rectangle(list(PAPUA_BBOX))
    img_t2 = s2_composite(cfg['periods']['t2'], papua)
    label  = build_label(papua, cfg['periods']['t2'])
    stack_t2 = img_t2.addBands(label.toFloat())
    tiles = make_tiles(papua, nx=cfg['export']['tiles_nx'], ny=cfg['export']['tiles_ny'])

    tasks = []
    for oti in corrupt_original:
        # 2. Hapus SEMUA file lama original tile ini di Drive (yang korup)
        old_files = list(TILES_T2.glob(f"papua_t2_tile_{oti:02d}-*.tif")) + \
                    list(TILES_T2.glob(f"papua_t2_tile_{oti:02d}.tif"))
        for of in old_files:
            of.unlink()
            print(f"  hapus file lama: {of.name}")

        # 3. Re-export tile ini (nama prefix sama → split otomatis oleh GEE)
        task = export_stack(
            stack_t2,
            description=f'papua_t2_tile_{oti:02d}',
            folder=TILES_T2.name,
            region=tiles[oti],
            scale=cfg['sentinel2']['scale'],
            max_pixels=int(cfg['export']['max_pixels']),
        )
        tasks.append(task)
        print(f"  -> re-export tile_{oti:02d} dimulai\n")

    # 4. Hapus folder patch + _DONE untuk sub-tile korup (biar di-cut ulang)
    for ti in corrupt_subtiles:
        d = PATCH_DIR / f"tile_{ti:03d}"
        if d.exists():
            shutil.rmtree(d, ignore_errors=True)
            print(f"  reset patch folder: tile_{ti:03d} (akan di-cut ulang)")

    print("\n" + "=" * 55)
    print(f"{len(tasks)} task re-export dimulai. Pantau di:")
    print("  https://code.earthengine.google.com/tasks")
    print("\nSetelah SEMUA task selesai (~15-30 menit):")
    print("  1. Jalankan ULANG Step 3 (cut patches) -> sub-tile korup di-cut ulang")
    print("  2. Jalankan cell AUDIT -> pastikan 144/144 komplit")

In [ ]:
# ── Cell 4b-3: Reset tile DARATAN yang ke-skip salah (_DONE=0 tapi ada data) ──
# Contoh: tile_118 punya 53% daratan tapi 0 patch karena dulu di-skip manual.
# Hapus folder+_DONE supaya Step 3 meng-cut-nya ulang dengan benar.
# (Butuh variabel low_subtiles dari Cell 4c-1 deteksi.)
import shutil

reset = []
for ti in low_subtiles:                       # low_subtiles = ada daratan (NaN < 95%)
    d = PATCH_DIR / f"tile_{ti:03d}"
    n = len(list(d.glob("p*.npz"))) if d.exists() else 0
    if n == 0:                                 # ada daratan TAPI 0 patch = salah skip
        shutil.rmtree(d, ignore_errors=True)
        reset.append(ti)
        print(f"tile_{ti:03d}: di-reset (daratan tapi 0 patch) -> akan di-cut ulang")

if not reset:
    print("Tidak ada tile daratan yang ke-skip salah. Aman.")
else:
    print(f"\nReset: {reset}. Sekarang jalankan ULANG Step 3, lalu Audit.")

tile_118: di-reset (daratan tapi 0 patch) -> akan di-cut ulang

Reset: [118]. Sekarang jalankan ULANG Step 3, lalu Audit.


In [ ]:
# ── Distribusi kelas: load dari cache JSON di Drive, atau hitung (paralel) ───
# Disimpan ke 'Satria Data 2.0/class_distribution.json' supaya tidak perlu
# hitung ulang tiap sesi. EDA & training cukup load file ini.
import json
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from forestwatch.data.patches import list_patches
from forestwatch.constants import CLASS_NAMES

DIST_PATH = DRIVE_ROOT / 'class_distribution.json'   # langsung di folder Satria Data 2.0

if DIST_PATH.exists():
    raw = json.load(open(DIST_PATH))
    dist = {int(k): int(v) for k, v in raw['counts'].items()}
    print(f"Distribusi dimuat dari cache: {DIST_PATH}")
    print(f"  (dihitung dari {raw.get('n_patches', '?')} patch)")
else:
    all_patches = list_patches(PATCH_DIR)
    print(f"Cache belum ada. Menghitung dari {len(all_patches):,} patch (24 worker paralel)...")
    counts = {c: 0 for c in range(6)}

    def _read_lab(p):
        return np.load(p)["lab"]

    with ThreadPoolExecutor(max_workers=24) as exe:
        for lab in tqdm(exe.map(_read_lab, all_patches),
                        total=len(all_patches), desc="Counting classes"):
            u, c = np.unique(lab, return_counts=True)
            for cls, cnt in zip(u.tolist(), c.tolist()):
                if 0 <= cls < 6:
                    counts[cls] += int(cnt)
    dist = counts

    json.dump(
        {'counts': {str(k): int(v) for k, v in dist.items()},
         'n_patches': len(all_patches)},
        open(DIST_PATH, 'w'), indent=2,
    )
    print(f"Distribusi disimpan ke: {DIST_PATH}")

total = sum(dist.values())
for cls in range(6):
    cnt = dist.get(cls, 0)
    pct = 100 * cnt / max(total, 1)
    print(f'  Kelas {cls} ({CLASS_NAMES[cls]:<16}): {cnt:>14,} piksel ({pct:5.2f}%)')
print(f'  Total                       : {total:>14,} piksel')

freqs = np.array([dist[c] / total for c in range(6)])
suggested = 1.0 / np.sqrt(freqs + 1e-9)
suggested = suggested / suggested.mean()
print('\nSuggested class_weights (1/sqrt(freq), normalized):', np.round(suggested, 2).tolist())


---
## Bagian 4c - Exploratory Data Analysis (EDA) Citra Satelit

Jalankan tiga cell ini **setelah** `cut_patches` selesai. EDA ini memvalidasi bahwa:

1. **Distribusi Kelas** — membuktikan class imbalance nyata di Papua dan justifikasi penggunaan class weights. Hutan mendominasi, Lahan Terbakar paling langka.
2. **Visualisasi Patch** — validasi visual langsung: apakah label fusion menghasilkan label yang masuk akal? Kita tumpangkan label berwarna di atas citra RGB. Jika label terlihat sesuai (hijau gelap = hutan, oranye = sawit), label fusion bekerja dengan baik.
3. **Spectral Signature per Kelas** — ini justifikasi ilmiah utama proyek: setiap kelas harus memiliki profil spektral berbeda di 6 band Sentinel-2. Jika grafik menunjukkan kurva yang saling terpisah (terutama di SWIR), model punya cukup informasi untuk membedakan kelas. Ini yang membuat deep learning valid untuk kasus ini.

Gambar dari EDA ini dapat langsung dimasukkan ke **Bagian Metodologi esai** sebagai bukti validasi data.

In [ ]:
# Load `dist` dari cache JSON di Drive kalau belum ada di memori (tahan restart sesi)
import json
if 'dist' not in globals():
    _raw = json.load(open(DRIVE_ROOT / 'class_distribution.json'))
    dist = {int(k): int(v) for k, v in _raw['counts'].items()}
    print('dist dimuat dari class_distribution.json')

# ── EDA 1: Distribusi Kelas ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from forestwatch.constants import CLASS_NAMES, CLASS_COLORS

# Hitung distribusi (pakai `dist` dari cell sebelumnya)
counts  = [dist.get(c, 0) for c in range(6)]
total   = sum(counts)
pcts    = [100 * c / total for c in counts]
colors  = [CLASS_COLORS[c] for c in range(6)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EDA 1 — Distribusi Kelas Tutupan Lahan Papua (Patch Training)',
             fontsize=13, fontweight='bold')

# Bar chart pixel count
bars = ax1.bar(CLASS_NAMES, counts, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Jumlah Piksel')
ax1.set_title('Pixel Count per Kelas')
ax1.tick_params(axis='x', rotation=30)
for bar, pct in zip(bars, pcts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
             f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

# Pie chart
wedges, texts, autotexts = ax2.pie(
    counts, labels=CLASS_NAMES, colors=colors,
    autopct='%1.1f%%', startangle=140,
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1}
)
ax2.set_title('Proporsi Kelas')

plt.tight_layout()
eda1_path = OUT_DIR / 'eda_class_distribution.png'
plt.savefig(eda1_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Disimpan: {eda1_path}')
print('\nCatatan untuk esai: Hutan mendominasi →',
      f'{pcts[1]:.1f}% piksel. Ini justifikasi class weights.')

In [ ]:
# ── EDA 2: Visualisasi Patch (RGB + Label) ───────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from forestwatch.constants import CLASS_NAMES, PALETTE_RGB
from forestwatch.data.patches import list_patches

patches = list_patches(PATCH_DIR)
rng     = np.random.default_rng(seed=42)
sample  = rng.choice(len(patches), size=min(12, len(patches)), replace=False)

fig, axes = plt.subplots(4, 6, figsize=(18, 12))
fig.suptitle('EDA 2 — Validasi Visual: Citra RGB vs Label Prediksi\n'
             '(Kiri: True Color RGB | Kanan: Label berwarna)',
             fontsize=13, fontweight='bold')

for row_i, patch_idx in enumerate(sample):
    data  = np.load(patches[patch_idx])
    img   = data['img']   # (6, H, W) float32 reflektansi [0,1]
    lab   = data['lab']   # (H, W) uint8 kelas 0-5

    # True color RGB: band B4(2), B3(1), B2(0) → index 2,1,0
    rgb = np.stack([img[2], img[1], img[0]], axis=-1)  # (H, W, 3)
    rgb = np.clip(rgb * 3.5, 0, 1)  # brighten untuk tampilan

    # Label berwarna
    label_rgb = np.zeros((*lab.shape, 3), dtype='uint8')
    for cls, color in PALETTE_RGB.items():
        label_rgb[lab == cls] = color

    col_pair = row_i * 2
    ax_img = axes[row_i // 3][col_pair % 6]
    ax_lbl = axes[row_i // 3][(col_pair + 1) % 6]

    ax_img.imshow(rgb)
    ax_img.set_title(f'Patch {patch_idx}\nRGB', fontsize=7)
    ax_img.axis('off')

    ax_lbl.imshow(label_rgb)
    ax_lbl.set_title(f'Label', fontsize=7)
    ax_lbl.axis('off')

# Legenda
legend_patches = [
    mpatches.Patch(color=[r/255, g/255, b/255], label=CLASS_NAMES[c])
    for c, (r, g, b) in PALETTE_RGB.items()
]
fig.legend(handles=legend_patches, loc='lower center', ncol=6,
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout(rect=[0, 0.04, 1, 1])
eda2_path = OUT_DIR / 'eda_sample_patches.png'
plt.savefig(eda2_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Disimpan: {eda2_path}')
print('Cek visual: apakah warna label sesuai dengan konten RGB?')

In [ ]:
# ── EDA 3: Spectral Signature per Kelas ─────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from forestwatch.constants import CLASS_NAMES, CLASS_COLORS, BANDS
from forestwatch.data.patches import list_patches

patches    = list_patches(PATCH_DIR)
N_SAMPLE   = min(3000, len(patches))  # sample untuk kecepatan
rng        = np.random.default_rng(seed=0)
sample_idx = rng.choice(len(patches), size=N_SAMPLE, replace=False)

# Akumulasi: sum + count per (kelas, band)
sum_per_class   = np.zeros((6, 6), dtype='float64')
count_per_class = np.zeros(6, dtype='float64')

for idx in sample_idx:
    data = np.load(patches[idx])
    img  = data['img']   # (6, H, W)
    lab  = data['lab']   # (H, W)
    for cls in range(6):
        mask = (lab == cls)
        n    = mask.sum()
        if n == 0:
            continue
        count_per_class[cls]  += n
        sum_per_class[cls]    += img[:, mask].sum(axis=1)

# Rata-rata reflektansi per kelas per band
mean_refl = np.zeros_like(sum_per_class)
for cls in range(6):
    if count_per_class[cls] > 0:
        mean_refl[cls] = sum_per_class[cls] / count_per_class[cls]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
band_labels = ['B2\nBlue', 'B3\nGreen', 'B4\nRed', 'B8\nNIR', 'B11\nSWIR1', 'B12\nSWIR2']
x = np.arange(6)

for cls in range(6):
    if count_per_class[cls] == 0:
        continue
    hex_color = CLASS_COLORS[cls]
    ax.plot(x, mean_refl[cls], marker='o', linewidth=2.5,
            color=hex_color, label=CLASS_NAMES[cls], markersize=7)

ax.set_xticks(x)
ax.set_xticklabels(band_labels, fontsize=10)
ax.set_ylabel('Rata-rata Reflektansi [0–1]', fontsize=11)
ax.set_xlabel('Band Sentinel-2', fontsize=11)
ax.set_title('EDA 3 — Spectral Signature per Kelas\n'
             '(Separabilitas spektral = model punya dasar membedakan kelas)',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)

# Annotasi penting
ax.axvspan(3.5, 5.5, alpha=0.07, color='red', label='_nolegend_')
ax.text(4.5, ax.get_ylim()[1] * 0.97, 'SWIR\n(pembeda\nsawit)', ha='center',
        fontsize=8, color='gray', va='top')

plt.tight_layout()
eda3_path = OUT_DIR / 'eda_spectral_signature.png'
plt.savefig(eda3_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Disimpan: {eda3_path}')
print('\nInterpretasi untuk esai:')
print('- Hutan (hijau gelap): NIR tinggi → vegetasi sehat (red-edge effect)')
print('- Sawit (oranye): SWIR berbeda dari Hutan → pola kanopi berbeda')
print('- Perairan (biru): semua band rendah, terutama NIR/SWIR')
print('- Lahan Terbakar (coklat): SWIR2 relatif tinggi (sisa bakar)')

---
## Bagian 5 - Build Model + Train (Minggu 2)

**Cell ini bisa jalan ~1-2 jam** di GPU T4 Colab dengan AMP. Pakai early stopping (patience 10) - bisa selesai lebih cepat.

**Peningkatan metodologi (berbasis literatur 2022-2025):**
- **Loss Focal+Tversky** (α=0.3, β=0.7): β>α menalti false-negative → recall kelas minoritas (Sawit, Lahan Terbakar) naik. Ganti `cfg['training']['loss']['type']` ke `ce_dice` untuk baseline lama.
- **Class weights median-frequency**: dihitung dari distribusi NYATA patch (bukan statis).
- **LR warmup 3 epoch** → cosine annealing (menstabilkan awal training).

**A/B Attention U-Net (opsional):** default `unet`. Untuk uji Attention, sebelum cell di bawah jalankan:
```python
cfg['model']['architecture'] = 'unet_scse'   # squeeze-excitation attention
```
lalu latih ulang dan bandingkan mIoU/Kappa dengan baseline (ablation study untuk esai).

In [ ]:
# Load `dist` dari cache JSON di Drive kalau belum ada di memori (tahan restart sesi)
import json
if 'dist' not in globals():
    _raw = json.load(open(DRIVE_ROOT / 'class_distribution.json'))
    dist = {int(k): int(v) for k, v in _raw['counts'].items()}
    print('dist dimuat dari class_distribution.json')

from forestwatch.data.dataset import build_dataloaders
from forestwatch.model.architecture import build_unet, count_parameters
from forestwatch.model.losses import make_loss_fn
from forestwatch.training.metrics import median_frequency_weights

train_loader, val_loader, test_loader = build_dataloaders(
    PATCH_DIR,
    batch_size=cfg['training']['batch_size'],
    num_workers=cfg['training']['num_workers'],
    train_ratio=cfg['training']['split']['train_ratio'],
    val_ratio=cfg['training']['split']['val_ratio'],
    seed=cfg['project']['seed'],
    augment_p=cfg['training']['augmentation'],
)
print(f'Split - Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}')

# Arsitektur: 'unet' (baseline) atau 'unet_scse' (Attention U-Net, untuk A/B).
# Default 'unet'. Untuk uji Attention: set cfg['model']['architecture']='unet_scse'.
model = build_unet(
    architecture=cfg['model']['architecture'],
    encoder_name=cfg['model']['encoder_name'],
    encoder_weights=cfg['model']['encoder_weights'],
    in_channels=cfg['model']['in_channels'],
    classes=cfg['model']['classes'],
)
print(f'Arsitektur: {cfg["model"]["architecture"]} | Parameter trainable: {count_parameters(model):,}')

# Class weights dari distribusi NYATA (median-frequency balancing), bukan statis.
# Butuh `dist` dari Bagian 4 (cell distribusi kelas).
if cfg['training'].get('use_class_weights', True):
    class_weights = median_frequency_weights(dist, n_classes=6)
    print('Class weights (median-freq):', class_weights)
else:
    class_weights = None

# Loss configurable - default Focal+Tversky (lihat configs/default.yaml)
lc = cfg['training']['loss']
loss_fn = make_loss_fn(
    loss_type=lc['type'],
    class_weights=class_weights,
    tversky_alpha=lc.get('tversky_alpha', 0.3),
    tversky_beta=lc.get('tversky_beta', 0.7),
    focal_gamma=lc.get('focal_gamma', 2.0),
)
print('Loss type:', lc['type'])

In [ ]:
from forestwatch.training.trainer import TrainConfig, train

tcfg = TrainConfig(
    epochs=cfg['training']['epochs'],
    patience=cfg['training']['patience'],
    learning_rate=cfg['training']['learning_rate'],
    weight_decay=cfg['training']['weight_decay'],
    amp=cfg['training']['amp'],
    warmup_epochs=cfg['training'].get('warmup_epochs', 3),
    seed=cfg['project']['seed'],
    ckpt_path=CKPT_PATH.as_posix(),
)
summary = train(model, train_loader, val_loader, loss_fn=loss_fn, cfg=tcfg)
print('\nTraining selesai:')
print(f"  best val mIoU: {summary['best_val_iou']:.4f} @ epoch {summary['best_epoch']}")
print(f"  ckpt        : {summary['ckpt_path']}")

In [ ]:
# Plot training history (loss + mIoU)
import matplotlib.pyplot as plt

hist = summary['history']
epochs = [h['epoch'] for h in hist]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, [h['train_loss'] for h in hist], label='train', linewidth=2)
ax1.plot(epochs, [h['val_loss'] for h in hist], label='val', linewidth=2)
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, [h['val_miou'] for h in hist], color='green', linewidth=2)
ax2.axhline(0.60, color='orange', linestyle='--', label='Target minimum (0.60)')
ax2.axhline(0.75, color='red', linestyle='--', label='Target ideal (0.75)')
ax2.set_title('Validation mIoU'); ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1)
ax2.legend(); ax2.grid(alpha=0.3)

fig.tight_layout()
fig_path = OUT_DIR / 'training_curve.png'
fig.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Training curve disimpan: {fig_path}')

---
## Bagian 6 - Evaluasi + Confusion Matrix + ONNX Export (Minggu 3)

In [ ]:
import torch
from forestwatch.training.trainer import evaluate
from forestwatch.training.metrics import compute_confusion_matrix, metric_summary
from forestwatch.utils.io import save_json

# Load checkpoint terbaik
model.load_state_dict(torch.load(CKPT_PATH, map_location='cpu'))
preds, targets = evaluate(model, test_loader)

cm = compute_confusion_matrix(preds, targets, n_classes=6)
metrics = metric_summary(cm, class_names=CLASS_NAMES)

print(f"Overall Accuracy: {metrics['overall_accuracy']*100:.2f}%")
print(f"Mean IoU       : {metrics['mean_iou']:.4f}")
print(f"Cohen's Kappa  : {metrics['kappa']:.4f}\n")
print(f"{'Kelas':<20}{'IoU':>10}{'F1':>10}")
print('-' * 40)
for row in metrics['per_class']:
    print(f"{row['class']:<20}{row['iou']:>10.4f}{row['f1']:>10.4f}")

metrics_path = OUT_DIR / 'metrics.json'
save_json(metrics, metrics_path)
print(f'\nmetrics.json disimpan: {metrics_path}')

In [ ]:
# Confusion matrix visualization (row-normalized)
import numpy as np
import matplotlib.pyplot as plt

cm_np = np.array(metrics['confusion_matrix'])
cm_norm = cm_np / cm_np.sum(axis=1, keepdims=True).clip(1)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=10,
                color='white' if cm_norm[i, j] > 0.5 else 'black')
ax.set_xticks(range(6)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(6)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix (row-normalized)')
fig.colorbar(im, ax=ax)
fig.tight_layout()
cm_path = OUT_DIR / 'confusion_matrix.png'
fig.savefig(cm_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Confusion matrix disimpan: {cm_path}')

In [ ]:
# Ekspor model.onnx (PRD §A.5 Cell 8)
from forestwatch.model.architecture import export_to_onnx

onnx_path = export_to_onnx(
    model,
    OUT_DIR / 'model.onnx',
    in_channels=cfg['model']['in_channels'],
    patch_size=cfg['inference']['patch_size'],
    opset_version=13,
)
print(f'model.onnx disimpan: {onnx_path}')

---
## Bagian 7 - Inferensi T1 + T2 (Minggu 3)

Jalankan model pada 36 ubin T1 + 36 ubin T2 dengan sliding window. Hasil: 72 GeoTIFF mask 1-band uint8 di `MASK_DIR`.

In [ ]:
from forestwatch.inference.tile_inference import infer_tiles_folder

model.eval()

# stride < patch_size = overlap-blending (hilangkan seam); tta = rata-rata 4 flip/rotasi
_stride = cfg['inference'].get('stride', cfg['inference']['patch_size'])
_tta = cfg['inference'].get('tta', False)
print(f'Inference: patch={cfg["inference"]["patch_size"]}, stride={_stride}, tta={_tta}')

infer_tiles_folder(
    tile_dir=TILES_T2,
    out_dir=MASK_DIR,
    model=model,
    prefix='mask_t2_',
    patch_size=cfg['inference']['patch_size'],
    stride=_stride,
    tta=_tta,
)

infer_tiles_folder(
    tile_dir=TILES_T1,
    out_dir=MASK_DIR,
    model=model,
    prefix='mask_t1_',
    patch_size=cfg['inference']['patch_size'],
    stride=_stride,
    tta=_tta,
)
print('Inferensi T1 + T2 selesai.')

---
## Bagian 8 - Generate 7 File Kontrak + Validasi Schema (Minggu 3)

Hasilkan semua file output sesuai [PRD §B.1](../../docs/PRD_ForestWatch_Papua_v2.md):
1. `landcover_2025.png` + `landcover_2025_bounds.json`
2. `landcover_2021.png` + `landcover_2021_bounds.json`
3. `deforestation.geojson` (4 transisi)
4. `statistics.json`
5. `legend.json`
6. `metrics.json`
7. `model.onnx` + `model_card.md`

In [ ]:
from forestwatch.outputs.orchestrator import generate_all_outputs
from forestwatch.utils.io import load_json

metrics = load_json(OUT_DIR / 'metrics.json')

paths = generate_all_outputs(
    mask_dir=MASK_DIR,
    out_dir=OUT_DIR,
    period_from=cfg['periods']['t1'],
    period_to=cfg['periods']['t2'],
    metrics=metrics,
    onnx_src=OUT_DIR / 'model.onnx',
    min_area_ha=cfg['change_detection']['min_area_ha'],
    model_card_kwargs=dict(
        epochs=cfg['training']['epochs'],
        batch_size=cfg['training']['batch_size'],
        n_parameters=count_parameters(model),
    ),
)
for k, v in paths.items():
    print(f'  {k:<25} {v}')

In [ ]:
# Validasi schema 7 file - WAJIB sebelum kirim ke Orang 2
from forestwatch.validation.schema import validate_outputs_dir

report = validate_outputs_dir(OUT_DIR)
print(report.render())
assert report.ok, 'Validasi gagal - perbaiki sebelum hands-off ke Orang 2.'

---
## Bagian 9 - Ringkasan Hasil untuk Tim (Esai + WebGIS)

Print angka kunci yang dipakai Orang 3 (esai) dan dikonfirmasi Orang 2 (WebGIS).

In [ ]:
stats = load_json(OUT_DIR / 'statistics.json')

print('=' * 60)
print('RINGKASAN HASIL - ForestWatch Papua')
print('=' * 60)
print(f"Periode pembanding   : {stats['period_from']} -> {stats['period_to']}")
print(f"Total deforestasi    : {stats['total_deforestation_ha']:>12,.1f} ha")
print(f"Jumlah hotspot       : {stats['n_hotspots']:>12,}")
print()
print('Per jenis transisi:')
for tname, ha in stats['per_transition_ha'].items():
    print(f"  {tname:<28} {ha:>12,.1f} ha")
print()
print('Per provinsi:')
for row in stats['per_province']:
    print(f"  {row['province']:<28} {row['deforestation_ha']:>12,.1f} ha")
print()
print('Per kelas (luas T2):')
for name, ha in stats['per_class_area_ha'].items():
    print(f"  {name:<28} {ha:>12,.1f} ha")
print()
print('Akurasi model:')
mm = stats['model_metrics']
print(f"  Overall Accuracy            {mm['overall_accuracy']*100:>11.2f}%")
print(f"  Mean IoU                    {mm['mean_iou']:>11.4f}")
if 'kappa' in mm:
    print(f"  Cohen's Kappa               {mm['kappa']:>11.4f}")
for row in mm['per_class']:
    print(f"  IoU {row['class']:<24} {row['iou']:>11.4f}")

In [ ]:
# Studi kasus Merauke - filter feature di Papua Selatan
import json
from collections import Counter

with open(OUT_DIR / 'deforestation.geojson') as f:
    fc = json.load(f)

merauke_feats = [
    ft for ft in fc['features']
    if ft.get('properties', {}).get('province') == 'Papua Selatan'
]
by_type = Counter(ft['properties']['transition_type'] for ft in merauke_feats)
area_by_type = {}
for ft in merauke_feats:
    t = ft['properties']['transition_type']
    area_by_type[t] = area_by_type.get(t, 0) + ft['properties']['area_ha']

print('STUDI KASUS MERAUKE (Papua Selatan)')
print('-' * 50)
print(f"Total hotspot   : {len(merauke_feats)}")
for t, n in by_type.most_common():
    print(f"  {t:<28} {n:>5} hotspot  ({area_by_type[t]:>10,.1f} ha)")

---
## Bagian 10 - Hands-off ke Tim

Aksi terakhir:

1. **Share folder `ForestWatch_Outputs/`** (Drive) dengan Orang 2 - akses Editor.
2. **Copy paste angka dari Bagian 9** ke pesan untuk Orang 3 (untuk esai).
3. **Backup notebook ini** ke GitHub repo tim.
4. **Update progress** di MASTER_PLAN.md - tick semua checklist Minggu 3.

**Selesai!** Stand by untuk Q&A juri di sesi presentasi. Pastikan menguasai semua angka di Bagian 9.